# Praktikum 1

## Teil 1: Datenbeschaffung & Transformation

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
import os
import shutil
from pyspark.sql.functions import col, when, count


spark = SparkSession.builder.appName("praktikum").master("local[*]").getOrCreate()

# OpenAQ Locations for csv download:
# Location_Id : City
# 2162185 : Ingolstadt
# 2669 Stachus

# Schema Definition, Umwandlung relevanter Spalten (Timestamp, Numerisch)
schema = StructType([
    StructField("location_id", IntegerType(), True),
    StructField("sensors_id", IntegerType(), True),
    StructField("location", StringType(), True),
    StructField("datetime", TimestampType(), True),
    StructField("lat", DoubleType(), True),
    StructField("lon", DoubleType(), True),
    StructField("parameter", StringType(), True),
    StructField("units", StringType(), True),
    StructField("value", DoubleType(), True),

])

# Einlesen
df1 = spark.read.schema(schema).option("header", "true").option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ssXXX").option("encoding", "UTF-8").csv("data/ingolstadt25")
df2 = spark.read.schema(schema).option("header", "true").option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ssXXX").option("encoding", "UTF-8").csv("data/münchen_stachus20")
df3 = spark.read.schema(schema).option("header", "true").option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ssXXX").option("encoding", "UTF-8").csv("data/ingolstadt26")
# one year contains about 20k data rows!
df4 = spark.read.schema(schema).option("header", "true").option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ssXXX").option("encoding", "UTF-8").csv("data/ingolstadt24")

df = df1.unionByName(df2).unionByName(df3).unionByName(df4)

print("Spark Dataframe Schema: \n")
print(df.printSchema())


print(df.count())
df.show(5, truncate=False)

# Behandlung fehlende Werte / Ausreißer
print("Missing Values / Outliers")
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()
# Interpretation: Missing Values fehlen schon im CSV, wie z.B. fehlender Dezember München 2020

# Ausreißer entfernen durch IQR
quantiles = df.approxQuantile("value", [0.25, 0.75], 0.01)
q1, q3 = quantiles
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
df = df.filter((df.value >= lower) & (df.value <= upper))

print("Bleibende Werte") 
print(df.count())



print("Overwrite parquet file")
if "data.parquet" in os.listdir("."):
    shutil.rmtree("data.parquet")
    
df.write.parquet("data.parquet")

## Wie entwickelt sich der Feinstaub Wert (pm10) für Ingolstadt in 2024 und 2025? 

In [ ]:
from pyspark.sql.functions import year, month
from pyspark.sql.functions import to_date, avg
import matplotlib.pyplot as plt

# Nur Ingolstadt, 2024 & 2025, nur pm10 herausgefiltert
df_2025_2024_pm10 = df.filter(
    (df.location == "Ingolstadt/Münchener Str.-2132151") &
    (df.parameter == "pm10") &
    (year(df.datetime).isin([2024, 2025])
)
)

print(df_2025_2024_pm10.count())
df_2025_2024_pm10.show(5)



# why is march 2024 so high? max value of 2070 ->  63 after outlier sampling
val_03 = df.filter( (df.location == "Ingolstadt/Münchener Str.-2132151") & (df.parameter == "pm10") & (year(df.datetime).isin([2024, 2025])) & (month(df.datetime) == 3) )
from pyspark.sql.functions import max
df.agg(max("value").alias("max_x")).show()
val_03.show(5)

In [ ]:
# Group data by months
df_monthly = df_2025_2024_pm10.groupBy(
    year("datetime").alias("year"),
    "month"
).agg(
    avg("value").alias("pm10_avg")
).orderBy("year", "month")



In [ ]:
# Plot data
pandas_df = df_monthly.toPandas()

for year in pdf["year"].unique():
    subset = pdf[pdf["year"] == year]
    plt.plot(subset["month"], subset["pm10_avg"], label=str(year))

plt.title("PM10 Entwicklung in Ingolstadt (2024 vs 2025)")
plt.xlabel("Monat")
plt.ylabel("PM10 Wert")
plt.legend()
plt.show()

# Interpretation PM10 für März/April 2024: Erst versucht, durch Ausreißer Sampling wegzubekommen, ohne Erfolg. Ausreißer sind
# aber entfernt worden, d.h. Wert muss natürlicher Form sein. Google sagt 45 als PM10 ist mäßig - hoch, aber nicht ungewöhnlich.


## Analytische Fragestellung 2